# 00 · Preparación del dataset para el modelo causal

**Proyecto:** El Sol Que Más Rinde — Estimación del retorno marginal del gasto público contra la anemia infantil
**Eje:** Finanzas y Gobernabilidad · Bootcamp de IA, PUCP
**Notebook:** `notebooks/03_causal_model/00_prep_dataset.ipynb`

### Objetivo de este notebook

Tomar el panel distrital ya construido (`data/clean/merged/panel_distrital_clean.csv`, producto de los notebooks `01_database/00a-00i` y el merge en `02_anemia_and_spending_models`) y dejarlo **listo para el modelo causal**: variables definidas por su rol (resultado, tratamiento, contexto territorial, controles), sin duplicados, con los nulos tratados de forma explícita y documentada, y con las inconsistencias conocidas del dato (encontradas en la auditoría EDA previa) corregidas o al menos declaradas.

Este notebook **no entrena ningún modelo**. Es el paso 0 de la carpeta `03_causal_model`:

```
00_prep_dataset.ipynb   ← este notebook: deja Y, T, X, W listos
01_causal_forest.ipynb  ← Double ML + Causal Forest, estima τ(X)
02_contrafractual.ipynb ← recalcula τ modificando una variable de X a la vez
```

### Principio de diseño

Cada decisión de limpieza en este notebook (qué filtrar, qué imputar, qué excluir) está **justificada con el número real encontrado en el dato**, no con un supuesto. Donde el dato tiene un problema conocido y no lo podemos resolver del todo aquí, lo declaramos explícitamente en vez de esconderlo — es la misma política de honestidad metodológica que exige el documento maestro del proyecto (sección 8, "Responsabilidad").


## 1. Fuente de datos y qué trae ya resuelto

El archivo de entrada es `data/clean/merged/panel_distrital_clean.csv`. Es la versión **ya corregida** del merge original (`panel_distrital_final.csv`), con tres correcciones que estaban pendientes en la bitácora del proyecto:

1. **Desfase de año de RENAMU corregido.** `personal_total` y `programa_anemia` son preguntas retrospectivas (describen el 31 de diciembre del año *anterior* a la encuesta). En esta versión ya están alineadas a `anio - 1`, en vez de quedar pegadas al año de la encuesta.
2. **Bug de `#¡NULO!` corregido.** En la versión anterior, las respuestas no registradas de RENAMU (marcadas `#¡NULO!` en el crudo) se habían decodificado como `0` ("No") en `programa_anemia` y `centro_salud_municipal`. Ahora quedan como `NaN` genuino — ya no se confunde "no respondió" con "respondió que no".
3. **Los 2 ubigeos sin geometría (`130112`, `180107`) fueron excluidos.** No tenían contexto territorial (no están en `geobase_distrital.gpkg`), así que no podían usarse en un modelo que depende justamente de ese contexto.

Lo que **todavía no está resuelto** en el archivo de entrada, y que este notebook trata explícitamente más abajo:

- El gasto (`gasto_total`, `gasto_anemia_pan`) sigue en soles absolutos, no per cápita.
- El `ubigeo` de SIAF identifica la unidad ejecutora del gasto, no necesariamente el distrito donde se presta el servicio — esto genera outliers administrativos que hay que tratar antes de estimar cualquier efecto.
- `region` no contiene costa/sierra/selva (es una copia del nombre del departamento) — no debe usarse como si fuera esa variable.
- El sesgo de selección del tamizaje de anemia (quién llega a hacerse la prueba) sigue sin corregir con ENDES — queda fuera del alcance de este notebook, es un modelo aparte.

Cada uno de estos puntos se aborda en su propia sección abajo, con el número real que aparece en el dato.


In [ ]:
# --- Imports y configuración ---
from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

# Bloque de rutas robusto (misma convención que 01_database/00a-00i):
# nunca depender de la ruta relativa desde la que se abrió Jupyter.
def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró '{marker}' subiendo desde {path}")

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"
INPUT_PATH = DATA / "clean" / "merged" / "panel_distrital_clean.csv"
OUTPUT_DIR = DATA / "clean" / "merged"

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"INPUT_PATH   = {INPUT_PATH}  (existe: {INPUT_PATH.exists()})")


## 2. Carga de datos

`ubigeo` se carga siempre como `string` — si se deja como entero, pandas descarta el cero a la izquierda de los departamentos 01-09 y la llave deja de calzar con el resto de tablas del proyecto (error ya documentado en la bitácora de `00a-00i`).


In [ ]:
df_raw = pd.read_csv(INPUT_PATH, dtype={"ubigeo": str})

print(f"Shape: {df_raw.shape}")
print(f"Distritos únicos (ubigeo): {df_raw['ubigeo'].nunique()}")
print(f"Rango de años: {df_raw['anio'].min()} - {df_raw['anio'].max()}")
df_raw.head(3)


## 3. Auditoría inicial

Antes de tocar nada, confirmamos programáticamente lo que dice la documentación del proyecto — no lo damos por sentado.


In [ ]:
# 3.1 Llave única
duplicados = df_raw.duplicated(subset=["ubigeo", "anio"]).sum()
print(f"Filas duplicadas en la llave (ubigeo, anio): {duplicados}")
assert duplicados == 0, "La llave ubigeo+anio debería ser única en el panel de entrada."

# 3.2 Formato de ubigeo
ubigeo_mal_formado = (~df_raw["ubigeo"].str.match(r"^\d{6}$")).sum()
print(f"Ubigeos que no son 6 dígitos: {ubigeo_mal_formado}")
assert ubigeo_mal_formado == 0

# 3.3 Nulos por columna (solo las que tienen algún nulo)
nulos = df_raw.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print("\nColumnas con nulos:")
print(nulos)


## 4. Ventana temporal de análisis

No todas las fuentes cubren los mismos años. `gasto_total`/`gasto_anemia_pan` (SIAF) están completos 2021-2025; `personal_total`/`programa_anemia` (RENAMU, ya corregidos a `anio - 1`) están completos sobre todo 2020-2023 y desaparecen en 2024-2026; `centro_salud_municipal` tiene un hueco fuerte en 2022 (herencia del bug de `#¡NULO!` que ahora sí se refleja como nulo real, no error).

La tabla de abajo se calcula directamente del dato — no es un número copiado de la documentación — para decidir la ventana con evidencia, no por default.


In [ ]:
cols_cobertura = [
    "ninos_evaluados", "prevalencia_anemia",
    "gasto_total", "gasto_anemia_pan",
    "personal_total", "programa_anemia", "centro_salud_municipal",
]
cobertura = df_raw.groupby("anio")[cols_cobertura].apply(lambda d: d.notna().sum())
print("Filas no nulas por año y variable (de un máximo de ~1889 distritos/año):\n")
print(cobertura.to_string())


In [ ]:
# Decisión: ventana 2021-2024.
# - gasto (SIAF) y anemia (SIEN) están prácticamente completos en los 4 años.
# - RENAMU (personal_total, programa_anemia) cubre bien 2021-2023 y cae a 0 en 2024;
#   se deja 2024 dentro de la ventana igual (para no perder gasto/anemia de ese año)
#   y el hueco de RENAMU en 2024 se resuelve con el indicador de "sin dato" + imputación
#   de la sección 6 — no se descarta el año completo por un solo grupo de variables.
ANIO_MIN, ANIO_MAX = 2021, 2024

panel = df_raw[df_raw["anio"].between(ANIO_MIN, ANIO_MAX)].copy()
print(f"Filas tras acotar a {ANIO_MIN}-{ANIO_MAX}: {len(panel)} ({panel['ubigeo'].nunique()} distritos)")


## 5. El marco causal: roles de las variables (Y, T, X, W)

Antes de limpiar nada más, fijamos qué papel cumple cada variable en el modelo Double ML + Causal Forest que se construye en `01_causal_forest.ipynb`. Esto no es una decisión de este notebook — viene directamente de la metodología del documento maestro (sección 5) — pero hay que declararla aquí porque determina qué tratamiento de nulos aplica a cada grupo.

| Rol | Significado | Variables |
|---|---|---|
| **Y** — resultado | Lo que queremos explicar | `prevalencia_anemia` (y sus insumos `ninos_evaluados`, `ninos_con_anemia`) |
| **T** — tratamiento | El gasto cuyo efecto marginal se quiere estimar | `gasto_total_percapita`, `gasto_anemia_pan_percapita` (y sus versiones log) |
| **X** — contexto territorial (eje de heterogeneidad) | Lo que el modelo de visión por computadora mide del territorio; es la variable respecto a la cual se estima **cómo cambia** el efecto del gasto | `pct_cultivo`, `pct_construido`, `pct_desnudo`, `pct_agua_visible`, `n_edificios`, `area_construida_m2`, `confianza_media`, `area_distrito_km2`, `densidad_edificios_km2`, `elevacion_media`, `pendiente_media`, `pct_agua_permanente`, `pct_agua_estacional`, `altitude`, `superficie`, `pob_densidad_2020` |
| **W** — controles de confusión | Variables que hay que controlar porque pueden confundir gasto↔anemia, pero que **no** son el eje de heterogeneidad territorial (son de gestión/administrativas, no de visión satelital) | `personal_total`, `programa_anemia`, `centro_salud_municipal`, `anio`, `macroregion_inei`, `macroregion_minsa` |

**Por qué X y W están separados:** el argumento central del proyecto es que el territorio (medido por visión por computadora) modula el rendimiento del gasto. Si mezcláramos la gestión municipal (W) dentro de X, el causal forest podría "explicar" heterogeneidad territorial con una variable que en realidad es de gestión administrativa, y la historia que cuenta el modelo dejaría de ser la que el proyecto se propone contar. `W` sigue entrando al modelo (como control, vía la función de nuisance de Double ML) — simplemente no es el eje sobre el que se reporta τ(X).


## 6. Controles de gestión municipal (W): nulos e indicadores de dato faltante

`personal_total`, `programa_anemia` y `centro_salud_municipal` vienen de RENAMU y **no** cubren todos los años de la ventana (ver tabla de la sección 4). Antes de imputar nada, la decisión de diseño es:

1. Crear una **bandera de "dato faltante"** por cada variable (`*_missing`), para que el modelo de ML de la etapa de nuisance (Double ML) pueda aprender que "no sabemos" es distinto de "sabemos que es bajo/que no aplica".
2. Imputar `personal_total` con la **mediana** del panel (variable continua, la mediana es robusta a los outliers de personal en distritos grandes).
3. Imputar `programa_anemia` y `centro_salud_municipal` con **0** — es una asunción conservadora explícita: ante la ausencia de reporte, no asumimos que el distrito sí tiene el programa o el centro de salud. Esto puede sesgar hacia abajo el efecto medido de estos controles en los años con más huecos (2022 sobre todo, por el hueco de `centro_salud_municipal`); queda declarado como limitación, no escondido.

Esta imputación es deliberadamente simple. Una alternativa más cuidadosa (imputación por modelo, o excluir por completo los años sin RENAMU) queda como trabajo futuro — no bloquea el núcleo mínimo demostrable del proyecto.


In [ ]:
w_cols = ["personal_total", "programa_anemia", "centro_salud_municipal"]

print("Nulos antes de imputar (dentro de la ventana 2021-2024):")
print(panel[w_cols].isna().sum())

for c in w_cols:
    panel[f"{c}_missing"] = panel[c].isna().astype(int)

mediana_personal = panel["personal_total"].median()
panel["personal_total"] = panel["personal_total"].fillna(mediana_personal)
panel["programa_anemia"] = panel["programa_anemia"].fillna(0)
panel["centro_salud_municipal"] = panel["centro_salud_municipal"].fillna(0)

print(f"\nMediana usada para personal_total: {mediana_personal:.1f}")
print("\nNulos después de imputar:")
print(panel[w_cols].isna().sum())
print("\nFilas marcadas como 'dato faltante' por variable:")
print(panel[[f'{c}_missing' for c in w_cols]].sum())


## 7. Filtro de tamaño de muestra mínimo (`ninos_evaluados`)

`prevalencia_anemia` es un promedio (`ninos_con_anemia / ninos_evaluados`). Cuando `ninos_evaluados` es muy chico (1, 2, 3 niños), esa proporción es casi puro ruido — un distrito con 1 niño evaluado y con anemia queda con `prevalencia_anemia = 1.0`, igual que un distrito con 500 niños evaluados y 500 con anemia, pero la evidencia detrás es completamente distinta. Lo mismo le pasa al gasto per cápita que se construye en la siguiente sección: dividir entre un denominador de 1 o 2 dispara el ratio a valores absurdos sin que el gasto en sí sea inusual.

Se aplica un umbral mínimo de **10 niños evaluados** por distrito-año para entrar al set analítico. No se descartan esas filas del archivo de origen — solo quedan fuera del dataset que alimenta el causal forest, y el conteo de cuántas se excluyen queda documentado abajo.


In [ ]:
MIN_NINOS_EVALUADOS = 10

n_antes_filtro_ninos = len(panel)
sin_dato_ninos = panel["ninos_evaluados"].isna().sum()
bajo_umbral = (panel["ninos_evaluados"] < MIN_NINOS_EVALUADOS).sum()  # incluye NaN como False, se cuenta aparte

panel_muestra_valida = panel[
    panel["ninos_evaluados"].notna() & (panel["ninos_evaluados"] >= MIN_NINOS_EVALUADOS)
].copy()

n_excluidas_filtro_ninos = n_antes_filtro_ninos - len(panel_muestra_valida)
print(f"Filas antes del filtro: {n_antes_filtro_ninos}")
print(f"  - sin dato de ninos_evaluados: {sin_dato_ninos}")
print(f"  - con ninos_evaluados < {MIN_NINOS_EVALUADOS}: {bajo_umbral}")
print(f"Filas después del filtro: {len(panel_muestra_valida)}  ({n_excluidas_filtro_ninos} excluidas)")
print(f"Distritos únicos que quedan: {panel_muestra_valida['ubigeo'].nunique()}")


## 8. Gasto per cápita y el sesgo de la unidad ejecutora administrativa

El diccionario de datos ya advierte que el `ubigeo` del gasto (SIAF) identifica **dónde está registrada la unidad ejecutora**, no necesariamente el distrito donde se presta el servicio. Al construir el gasto per cápita (`gasto_total / ninos_evaluados`) ese problema se vuelve visible y medible: hay distritos — típicamente sedes administrativas centrales — donde el gasto per cápita es cientos de veces el de un distrito típico, no porque ahí se invierta más en la población local, sino porque ahí está la sede que administra presupuesto de todo un ámbito mucho más amplio.

Al calcular el percentil 99 del gasto total per cápita sobre el panel filtrado, aparecen casos como estos (verificado directamente en el dato, no es un ejemplo hipotético):

- **Lima (150101)**, sede de unidades ejecutoras nacionales de salud: gasto total anual de hasta S/ 71 mil millones un solo año — un orden de magnitud que no tiene sentido como "gasto en Lima Cercado".
- **San Isidro (150131)**: hasta S/ 18 mil millones/año con apenas ~170-280 niños evaluados, dando un gasto per cápita de más de S/ 100 millones por niño.
- Otros distritos con el mismo patrón: **Miraflores, San Borja, Jesús María** (Lima) y **Arequipa** (capital de provincia) — todos sedes de unidades ejecutoras regionales o nacionales, no destinos reales del gasto que miden.

Incluir estos casos en el modelo causal introduciría una relación completamente espuria entre "gasto" y "anemia" que no refleja ninguna intervención real sobre esos niños. La decisión: **marcar como outlier administrativo todo distrito-año por encima del percentil 99 de gasto total per cápita, y excluirlo del set analítico principal** — pero conservarlo aparte, en un archivo separado, para que quede trazable y no se pierda silenciosamente.


In [ ]:
panel_gasto = panel_muestra_valida[panel_muestra_valida["gasto_total"].notna()].copy()

panel_gasto["gasto_total_percapita"] = panel_gasto["gasto_total"] / panel_gasto["ninos_evaluados"]
panel_gasto["gasto_anemia_pan_percapita"] = panel_gasto["gasto_anemia_pan"] / panel_gasto["ninos_evaluados"]
panel_gasto["log_gasto_total_percapita"] = np.log1p(panel_gasto["gasto_total_percapita"])
panel_gasto["log_gasto_anemia_pan_percapita"] = np.log1p(panel_gasto["gasto_anemia_pan_percapita"])

p99_gasto = panel_gasto["gasto_total_percapita"].quantile(0.99)
panel_gasto["flag_gasto_outlier_administrativo"] = panel_gasto["gasto_total_percapita"] > p99_gasto

n_outliers = panel_gasto["flag_gasto_outlier_administrativo"].sum()
distritos_outliers = sorted(
    panel_gasto.loc[panel_gasto["flag_gasto_outlier_administrativo"], "distrito"].dropna().unique()
)

print(f"Umbral p99 de gasto_total_percapita: S/ {p99_gasto:,.0f} por niño evaluado")
print(f"Filas marcadas como outlier administrativo: {n_outliers} de {len(panel_gasto)}")
print(f"Distritos afectados ({len(distritos_outliers)}): {distritos_outliers}")

distritos_excluidos_admin = panel_gasto[panel_gasto["flag_gasto_outlier_administrativo"]].copy()
panel_sin_outliers = panel_gasto[~panel_gasto["flag_gasto_outlier_administrativo"]].copy()
print(f"\nFilas que quedan tras excluir outliers administrativos: {len(panel_sin_outliers)}")


## 9. Variable de resultado (Y): validación final

Filtramos las filas sin `prevalencia_anemia` (no se puede entrenar el modelo sin resultado observado). Esto es aparte del filtro de la sección 7 porque un distrito puede tener `ninos_evaluados >= 10` pero, en algún caso residual del merge, no traer `prevalencia_anemia` calculada.

**Limitación que este notebook no resuelve, y que queda declarada:** `prevalencia_anemia` viene del registro administrativo de tamizajes (SIEN), no de un censo — refleja quién *llegó* a tamizarse, no necesariamente la prevalencia real del distrito. La corrección de este sesgo de selección con ENDES es un modelo aparte (bayesiano de dos cabezas, según la metodología del proyecto), todavía no incorporado al pipeline. El dataset que se guarda aquí trabaja con la prevalencia administrativa tal como está, y ese hecho debe repetirse en cualquier interpretación de τ que se haga más adelante.


In [ ]:
n_antes_filtro_y = len(panel_sin_outliers)
panel_final = panel_sin_outliers[panel_sin_outliers["prevalencia_anemia"].notna()].copy()
n_excluidas_filtro_y = n_antes_filtro_y - len(panel_final)
print(f"Filas sin prevalencia_anemia excluidas: {n_excluidas_filtro_y}")
print(f"Filas finales: {len(panel_final)}  |  Distritos únicos: {panel_final['ubigeo'].nunique()}")


## 10. Nota sobre `region` — no es costa/sierra/selva

El diccionario de datos original describe `region` como "región geográfica (costa/sierra/selva u homóloga)". En la práctica, la columna es una copia literal de `departamento` (26 valores únicos, uno por departamento — no 3 categorías naturales). No es un error de este notebook ni del merge: así se generó desde `00a_geobase.ipynb`.

Para no arrastrar una variable que promete algo que no tiene, `region` **no se incluye** en el set de controles (W) — se usa en su lugar `macroregion_inei` y `macroregion_minsa`, que sí son agrupaciones agregadas (no una copia 1:1 de `departamento`). Construir la clasificación real de costa/sierra/selva (hay tablas públicas de INEI para eso) queda como trabajo futuro, fuera del alcance de este notebook.


In [ ]:
print("Valores únicos de 'region' (debería ser ~3, en la práctica son los departamentos):")
print(panel_final["region"].nunique(), "valores únicos —", sorted(panel_final["region"].dropna().unique())[:6], "...")
print()
print("Valores únicos de 'macroregion_inei':", sorted(panel_final["macroregion_inei"].dropna().unique()))
print("Valores únicos de 'macroregion_minsa':", sorted(panel_final["macroregion_minsa"].dropna().unique()))


## 11. Selección y organización final de columnas

Se arma el dataframe final con las columnas agrupadas por rol (llaves → identificación → Y → T → X → W → banderas de calidad), en ese orden, para que `01_causal_forest.ipynb` pueda simplemente hacer slicing por bloque sin tener que redescubrir qué es cada columna.


In [ ]:
cols_llave = ["ubigeo", "anio"]
cols_identificacion = ["departamento", "provincia", "distrito"]

cols_Y = ["ninos_evaluados", "ninos_con_anemia", "prevalencia_anemia"]

cols_T = [
    "gasto_total", "gasto_anemia_pan",
    "gasto_total_percapita", "gasto_anemia_pan_percapita",
    "log_gasto_total_percapita", "log_gasto_anemia_pan_percapita",
]

cols_X = [
    "pct_cultivo", "pct_construido", "pct_desnudo", "pct_agua_visible",
    "n_edificios", "area_construida_m2", "confianza_media",
    "area_distrito_km2", "densidad_edificios_km2",
    "elevacion_media", "pendiente_media",
    "pct_agua_permanente", "pct_agua_estacional",
    "altitude", "superficie", "pob_densidad_2020",
]

cols_W = [
    "personal_total", "personal_total_missing",
    "programa_anemia", "programa_anemia_missing",
    "centro_salud_municipal", "centro_salud_municipal_missing",
    "anio", "macroregion_inei", "macroregion_minsa",
]

cols_finales = cols_llave + cols_identificacion + cols_Y + cols_T + cols_X + cols_W

# anio ya está en cols_llave; se referencia también en W (rol doble: llave e insumo
# de control por año) pero no se duplica la columna en el dataframe final.
cols_finales_unicas = list(dict.fromkeys(cols_finales))

panel_causal = panel_final[cols_finales_unicas].copy()
print(f"Columnas finales ({len(panel_causal.columns)}): {list(panel_causal.columns)}")


## 12. Distritos con contexto territorial incompleto en origen

El diccionario de datos ya documenta que 15 distritos —en su mayoría zonas remotas de selva/sierra (Ahuayro, Putis, Cielo Punco, Unión Asháninka, entre otros)— llegaron **sin** `altitude`, `superficie` ni `pob_densidad_2020` desde el propio `geobase_distrital.gpkg` de origen. No es un error de este pipeline: es un vacío real de la fuente censal/geográfica para esos distritos.

Como estas tres columnas son parte de **X**, el eje de heterogeneidad territorial, no tiene sentido inventarles un valor por imputación — el argumento completo del proyecto es que X mide el territorio real. La política declarada en el documento maestro (sección 8) es justamente esta: **declarar los distritos con datos insuficientes en vez de inventar un valor.** Se excluyen del set analítico y se listan aquí explícitamente.


In [ ]:
cols_X_con_nulos_origen = ["altitude", "superficie", "pob_densidad_2020"]

mask_x_incompleto = panel_causal[cols_X_con_nulos_origen].isna().any(axis=1)
distritos_x_incompleto = sorted(
    panel_causal.loc[mask_x_incompleto, "distrito"].dropna().unique()
)

print(f"Filas excluidas por contexto territorial incompleto en origen: {mask_x_incompleto.sum()}")
print(f"Distritos afectados ({len(distritos_x_incompleto)}): {distritos_x_incompleto}")

panel_causal = panel_causal.loc[~mask_x_incompleto].copy()
print(f"\nFilas que quedan: {len(panel_causal)}  |  Distritos únicos: {panel_causal['ubigeo'].nunique()}")


## 13. Validaciones finales antes de guardar

Antes de escribir el archivo a disco, se repiten las validaciones de la sección 3 sobre el dataset final, más una verificación de que ninguna columna de Y, T o X (las que el causal forest necesita sí o sí) tiene nulos.


In [ ]:
assert panel_causal.duplicated(subset=["ubigeo", "anio"]).sum() == 0, "Duplicados en la llave"

nulos_criticos = panel_causal[cols_Y + cols_T + cols_X].isna().sum()
nulos_criticos = nulos_criticos[nulos_criticos > 0]
assert nulos_criticos.empty, f"Hay nulos en columnas críticas (Y/T/X):\n{nulos_criticos}"

print("✅ Sin duplicados en la llave (ubigeo, anio)")
print("✅ Sin nulos en las columnas de Y, T y X")
print(f"\nShape final: {panel_causal.shape}")
print(f"Distritos únicos: {panel_causal['ubigeo'].nunique()}")
print(f"Años cubiertos: {sorted(panel_causal['anio'].unique())}")
print()
panel_causal[cols_Y + ["gasto_total_percapita", "log_gasto_total_percapita"]].describe()


## 14. Guardado

Se guardan tres archivos, todos dentro de `data/clean/merged/` (carpeta versionada en git, a diferencia de `data/raw/`):

1. **`panel_causal_ready.csv`** — el dataset listo para `01_causal_forest.ipynb`.
2. **`panel_causal_excluidos_administrativos.csv`** — las filas excluidas por ser outliers administrativos de gasto (sección 8), conservadas para trazabilidad, no para modelar.
3. **`panel_causal_ready_metadata.json`** — un resumen reproducible de cada filtro aplicado (cuántas filas entraron/salieron y por qué), para que la pantalla de metodología y limitaciones de la app final pueda citar estos números sin tener que volver a correr el notebook.


In [ ]:
OUTPUT_PATH = OUTPUT_DIR / "panel_causal_ready.csv"
OUTPUT_EXCLUIDOS_PATH = OUTPUT_DIR / "panel_causal_excluidos_administrativos.csv"
OUTPUT_METADATA_PATH = OUTPUT_DIR / "panel_causal_ready_metadata.json"

panel_causal.to_csv(OUTPUT_PATH, index=False)
distritos_excluidos_admin.to_csv(OUTPUT_EXCLUIDOS_PATH, index=False)

metadata = {
    "generado_en": datetime.now(timezone.utc).isoformat(),
    "fuente": str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    "ventana_temporal": {"anio_min": ANIO_MIN, "anio_max": ANIO_MAX},
    "filas_fuente_en_ventana": int(len(panel)),
    "filtro_ninos_evaluados": {
        "umbral_minimo": MIN_NINOS_EVALUADOS,
        "filas_excluidas": int(n_excluidas_filtro_ninos),
    },
    "outliers_administrativos_gasto": {
        "percentil_usado": 0.99,
        "umbral_soles_percapita": float(p99_gasto),
        "filas_excluidas": int(n_outliers),
        "distritos_afectados": distritos_outliers,
    },
    "filas_sin_prevalencia_anemia_excluidas": int(n_excluidas_filtro_y),
    "contexto_territorial_incompleto_en_origen": {
        "filas_excluidas": int(mask_x_incompleto.sum()),
        "distritos_afectados": distritos_x_incompleto,
    },
    "imputacion_renamu": {
        "personal_total": {"metodo": "mediana", "valor": float(mediana_personal)},
        "programa_anemia": {"metodo": "constante", "valor": 0},
        "centro_salud_municipal": {"metodo": "constante", "valor": 0},
    },
    "shape_final": {"filas": int(panel_causal.shape[0]), "columnas": int(panel_causal.shape[1])},
    "distritos_unicos_final": int(panel_causal["ubigeo"].nunique()),
    "anios_final": sorted(int(a) for a in panel_causal["anio"].unique()),
    "roles_de_variables": {
        "Y": cols_Y,
        "T": cols_T,
        "X": cols_X,
        "W": cols_W,
        "llave": cols_llave,
    },
    "limitaciones_declaradas": [
        "prevalencia_anemia viene de registro administrativo (SIEN), no de un censo; "
        "sesgo de selección de quién se tamiza todavía no corregido con ENDES.",
        "El ubigeo del gasto (SIAF) identifica la unidad ejecutora, no necesariamente "
        "el distrito de intervención real; se trató el caso extremo (outliers "
        "administrativos) pero el proxy sigue siendo imperfecto para el resto de distritos.",
        "gasto_anemia_pan solo cubre el Programa Articulado Nutricional (0001), no "
        "salud materno-neonatal, saneamiento rural, Cuna Más, Qali Warma, JUNTOS ni "
        "incentivos municipales.",
        "region no es costa/sierra/selva (es una copia de departamento); no se usó "
        "como variable de control por esa razón.",
        "La imputación de programa_anemia/centro_salud_municipal con 0 ante dato "
        "faltante es una asunción conservadora explícita, no un hecho verificado.",
    ],
}

with open(OUTPUT_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Guardado: {OUTPUT_PATH.relative_to(PROJECT_ROOT)}  ({len(panel_causal)} filas)")
print(f"Guardado: {OUTPUT_EXCLUIDOS_PATH.relative_to(PROJECT_ROOT)}  ({len(distritos_excluidos_admin)} filas)")
print(f"Guardado: {OUTPUT_METADATA_PATH.relative_to(PROJECT_ROOT)}")


## 15. Resumen y siguiente paso

En una corrida típica, este notebook parte de ~13,100 filas del panel completo (2020-2026) y termina con un dataset de análisis acotado a 2021-2024, con la muestra mínima de niños evaluados, sin outliers administrativos de gasto, y sin nulos en las variables Y/T/X que el causal forest necesita. Las cifras exactas de la corrida quedan impresas arriba y guardadas en `panel_causal_ready_metadata.json` — cítense esas, no las de este párrafo, si cambia el dato de entrada.

**Siguiente notebook: `01_causal_forest.ipynb`.** Debe:

1. Cargar `panel_causal_ready.csv`.
2. Usar `cols_T` (gasto, de preferencia la versión log per cápita) como tratamiento, `cols_Y` como resultado, `cols_X` como eje de heterogeneidad, `cols_W` como controles de nuisance — los cuatro bloques ya vienen documentados en `panel_causal_ready_metadata.json → roles_de_variables`.
3. Aplicar Double ML (cross-fitting) para residualizar T e Y contra W y X, y un Causal Forest con honest splitting sobre los residuos para estimar τ(X) con intervalo de confianza por distrito-año.
4. Recordar, al reportar τ, que es un **índice de priorización basado en heterogeneidad estimada**, no un efecto causal probado — no hay todavía una fuente de variación exógena confirmada (el documento maestro menciona el canon minero como instrumento a explorar, sección 5).
